### Libraries

In [5]:
import os
import chromadb
from sentence_transformers import SentenceTransformer
from openai import OpenAI

### Load Codebase

Change "./repo" to your project folder.

In [6]:
def load_codebase(repo_path):

    documents=[]

    for root,dirs,files in os.walk(repo_path):

        for file in files:

            if file.endswith((".py",".js",".ts",".java",".go",".cpp",".cs")):

                path=os.path.join(root,file)

                with open(path,"r",encoding="utf-8",errors="ignore") as f:
                    content=f.read()

                documents.append({
                    "file":path,
                    "content":content
                })

    return documents


docs=load_codebase("./community-app")

print("Files loaded:",len(docs))

Files loaded: 16


### Chunk Code

We split large files into smaller parts.

In [7]:
def chunk_text(text,chunk_size=500):

    chunks=[]

    for i in range(0,len(text),chunk_size):
        chunks.append(text[i:i+chunk_size])

    return chunks

### Create Chunks Dataset

In [8]:
chunks=[]

for doc in docs:

    for chunk in chunk_text(doc["content"]):

        chunks.append({
            "file":doc["file"],
            "content":chunk
        })

print("Total chunks:",len(chunks))

Total chunks: 31


### Load Embedding Model

Using SentenceTransformers

In [9]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Create Chroma Database

In [10]:
client = chromadb.Client()

collection = client.create_collection("codebase")

### Insert Code Chunks Into Chroma

In [ ]:
documents=[c["content"] for c in chunks]

metadatas=[{"file":c["file"]} for c in chunks]

ids=[str(i) for i in range(len(chunks))]

embeddings=embedding_model.encode(documents).tolist()


collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print("Chunks stored:",len(documents))

Chunks stored: 31


### Search Function

In [ ]:
# def search_code(query,k=5):

#     query_embedding = embedding_model.encode(query).tolist()

#     results = collection.query(
#         query_embeddings=[query_embedding],
#         n_results=k
#     )

#     return results
def search_code(query,k=3):  # reduce k
    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results

### Test Retrieval

In [13]:
results = search_code("authentication login function")

for doc,meta in zip(results["documents"][0],results["metadatas"][0]):

    print("File:",meta["file"])
    print(doc[:200])
    print("------")

File: ./community-app/src/middleware/auths.ts

import { Request, Response, NextFunction } from 'express';
import { verifyToken } from '../Utils/tokenize';
import { UserResponse } from '../interface/user';

export const authMiddleware = (req: Requ
------
File: ./community-app/src/routes/auth.routes.ts
import { Router } from 'express';
import { login, register } from '../controller/auth';

const router = Router();

router.post('/register', register);
router.post('/login', login);

export default rou
------
File: ./community-app/src/interface/auth.ts
// auths and payloads

export interface RegisterInput {
    email: string;
    password: string;
    confirmPassword: string;
  }
  
  export interface LoginInput {
    email: string;
    password: st
------
File: ./community-app/src/Utils/Hasher.ts
import bcrypt from 'bcryptjs';

export const hashPassword = async (password: string) => {
  return await bcrypt.hash(password, 10);
};

export const comparePassword = async (password: string, hash

### Setup LLM

Using OpenRouter

In [14]:

client_llm = OpenAI(
    base_url=os.getenv("OPENROUTER_BASE_URL"),
    api_key=os.getenv("OPENROUTER_API_KEY")
)

### Create AI Answer Function

In [15]:
def ask_codebase(question):

    results=search_code(question)

    context="\n\n".join(results["documents"][0])

    prompt=f"""
Answer the question using the code context.

Question:
{question}

Code Context:
{context}

Explain clearly and mention file names.
"""

    response=client_llm.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[{"role":"user","content":prompt}]
    )

    return response.choices[0].message.content

### Ask Questions

In [16]:
ask_codebase("Where is authentication implemented?")

"Authentication is implemented in the following files:\n\n1. **Middleware File**:\n   - **File Name**: The specific filename isn't mentioned, but it contains the `authMiddleware` function.\n   - **Implementation Details**: The `authMiddleware` function handles the authentication logic by checking for an `Authorization` header in incoming requests. It expects a Bearer token format, extracts the token, and then uses the `verifyToken` utility function to validate it. If the token is valid, the user information is attached to the request object, and the next middleware is called. If the token is invalid or missing, it responds with a 401 Unauthorized status.\n\n2. **Utility File**:\n   - **File Name**: Not explicitly mentioned, but it includes the `generateToken` and `verifyToken` functions.\n   - **Implementation Details**: \n     - The `generateToken` function creates a new JWT token based on the user's information (id, email, role) and signs it using a secret key.\n     - The `verifyTok

### Interactive Chat

In [ ]:
while True:

    q=input("Ask about the codebase: ")

    if q.lower()=="exit":
        break

    print("\nAI Answer:\n")

    print(ask_codebase(q))

    print("\n")


AI Answer:

Based on the provided code context and snippets, here's a clear explanation of the relevant files and their functionalities:

1. **Controller (assumed filename: `applicationController.ts`)**:
   - This file contains the logic for handling application responses. 
   - The `respondToApplication` function processes a request to update the status of an application.
     ```javascript
     const { applicationId, action } = req.body;

     if (!applicationId || !["APPROVED", "REJECTED"].includes(action)) {
         res.status(400).json({ message: "Invalid request" });
     }

     const updatedApp = await prisma.application.update({
         where: { id: applicationId },
         data: { status: action },
     });

     res.status(200).json({
         message: `Application ${action.toLowerCase()}`,
         updatedApplication: updatedApp,
     });
     ```
   - This extracts the `applicationId` and `action` (which should be either "APPROVED" or "REJECTED") from the request body.